# Proyecto integrador YOLO26 — Notebook 02: Detección, segmentación y pose sobre video
Integra simultáneamente YOLO26 Detection personalizado (`mate`, `termo`), YOLO26 Segmentation COCO sin personas y YOLO26 Pose para personas. Produce un solo `.mp4` por video original.

**Requisitos previos:** notebook 01 ejecutado, `best.pt` existente, y 2–3 videos públicos **descargados legítimamente** donde aparezcan las clases, personas y otros objetos COCO. Guardar fuente y licencia de cada video. No se exige que todos los fotogramas tengan las tres tareas visibles.

## 1. Entorno y rutas compartidas

In [ ]:
%pip -q install -U ultralytics opencv-python-headless pandas matplotlib pillow

In [ ]:
import sys, os, json, subprocess, shutil, time
from pathlib import Path
from collections import Counter
import numpy as np, cv2, pandas as pd, torch, matplotlib.pyplot as plt
from IPython.display import display, Video, Image as IPyImage
from ultralytics import YOLO

# 1. Detección automática de entorno (Google Colab vs Local)
IN_COLAB = 'google.colab' in sys.modules or os.path.exists('/content')

if IN_COLAB:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        DRIVE_ROOT = Path('/content/drive/MyDrive/Proyecto_Integrador_YOLO26')
        ROOT = DRIVE_ROOT if DRIVE_ROOT.exists() else Path('/content/CV-mate-detection')
    except Exception as e:
        print('Advertencia en Colab Drive mount:', e)
        ROOT = Path('/content/CV-mate-detection') if Path('/content/CV-mate-detection').exists() else Path.cwd()
else:
    ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()

# Rutas de modelos y videos
BEST = ROOT / 'entrenamientos' / 'mate_termo_yolo26' / 'weights' / 'best.pt'
if not BEST.exists():
    alt_best = ROOT / 'models' / 'best.pt'
    if alt_best.exists():
        BEST = alt_best

VIDEOS_IN = ROOT / 'videos' / 'input'
VIDEOS_OUT = ROOT / 'videos' / 'output'

VIDEOS_OUT.mkdir(parents=True, exist_ok=True)
VIDEOS_IN.mkdir(parents=True, exist_ok=True)

DEVICE = 0 if torch.cuda.is_available() else 'cpu'

print('=== ENTORNO DE EJECUCIÓN ===')
print('Modo Colab:', IN_COLAB)
print('Directorio raíz:', ROOT)
print('Device:', DEVICE)
print('Ruta pesos custom (BEST):', BEST)
print('Existe best.pt:', BEST.exists())
if not BEST.exists():
    print('[AVISO]: best.pt aún no ha sido entrenado. Entrenar primero ejecutando el Notebook 01.')


## 2. Fuentes de videos y verificaciones
Descargar manualmente videos públicos con permiso de uso (por ejemplo Pexels) y subir `.mp4`, `.mov`, `.mkv` a `videos_originales/`. Anotar URL, licencia y fecha en `fuentes_videos.csv`. La consigna permite hasta 2 minutos por video y recomienda 2–3 videos de ~20 segundos.

**Criterio de selección:** al menos un video con mate/termo visibles y personas para pose; además, debe haber objetos COCO distintos de personas para ver máscaras (mesa, silla, vaso, etc.). No hay garantías de que cualquier video descargado los contenga. Es preferible comprobarlo antes de finalizar la entrega.

In [ ]:
VIDEO_EXT={'.mp4','.mov','.mkv','.avi','.webm'}
videos=sorted(p for p in VIDEOS_IN.iterdir() if p.is_file() and p.suffix.lower() in VIDEO_EXT)
if not videos:
    raise FileNotFoundError(f'No hay videos de entrada. Subir 2–3 videos a {VIDEOS_IN}.')
rows=[]
for p in videos:
    cap=cv2.VideoCapture(str(p)); fps=cap.get(cv2.CAP_PROP_FPS); frames=cap.get(cv2.CAP_PROP_FRAME_COUNT)
    width=cap.get(cv2.CAP_PROP_FRAME_WIDTH); height=cap.get(cv2.CAP_PROP_FRAME_HEIGHT)
    cap.release(); duration=frames/fps if fps>0 else -1
    rows.append({'video':p.name,'segundos':round(duration,1),'fps':round(fps,2),'resolution':f'{int(width)}x{int(height)}'})
info=pd.DataFrame(rows); display(info)
if (info.segundos<=0).any() or (info.segundos>120).any():
    raise ValueError('Hay videos ilegibles o de más de 120 segundos. Revisar los archivos.')

## 3. Tres modelos, umbrales y reglas
- `best.pt`: únicamente `mate` y `termo`, dibujados como **bounding boxes** con confianza.
- `yolo26n-seg.pt`: instancias COCO excepto `person`. Si una máscara coincide en gran proporción con una caja personalizada (por ejemplo, `bottle` confunde un termo), se omite esa máscara.
- `yolo26n-pose.pt`: keypoints y líneas del esqueleto humano, con seguimiento temporal por ID cuando se disponga de identificadores del tracker.

El uso de modelos preentrenados para Segmentación y Pose cumple la integración; solo entrenamos Detection.

In [ ]:
if not BEST.exists():
    raise FileNotFoundError(f'Falta best.pt en {BEST}. Ejecutar primero el Notebook 01 para entrenar el modelo.')

DETECTOR = YOLO(str(BEST))

# Modelos oficiales preentrenados en COCO (compatibles con el enunciado de YOLO26)
SEG_MODEL_NAME = 'yolo11n-seg.pt' if not Path('yolo26n-seg.pt').exists() else 'yolo26n-seg.pt'
POSE_MODEL_NAME = 'yolo11n-pose.pt' if not Path('yolo26n-pose.pt').exists() else 'yolo26n-pose.pt'

SEGMENTADOR = YOLO(SEG_MODEL_NAME)
POSE = YOLO(POSE_MODEL_NAME)

CONF_CUSTOM = 0.35
CONF_SEG = 0.45
CONF_POSE = 0.35
CONF_KEYPOINT = 0.35
MASK_ALPHA = 0.35
MASK_OVERLAP_SUPPRESS = 0.5  # % del área de la máscara dentro de alguna caja personalizada
IMGSZ = 640
MAX_SECONDS = 120

assert DETECTOR.task == 'detect' and SEGMENTADOR.task == 'segment' and POSE.task == 'pose'
print('Clases del detector personalizado:', DETECTOR.names)
print('Modelos cargados exitosamente:')
print('  - Detección custom:', BEST.name)
print('  - Segmentación COCO:', SEG_MODEL_NAME)
print('  - Pose estimation:', POSE_MODEL_NAME)


## 4. Funciones de superposición de máscaras, boxes y pose
La salida es un fotograma BGR. Se filtra la clase COCO `person` mediante ID identificado por nombre. La supresión de máscaras para clases nuevas es una heurística que debe revisarse visualmente (la confusión termo/botella puede ser frecuente).

In [ ]:
COLORES_CUSTOM={'mate':(30,230,30),'termo':(40,160,255)}
COLORES_SEG=[(205,130,40),(180,60,180),(40,170,170),(110,210,60),(200,80,80)]
COCO_SKELETON=[(0,1),(0,2),(1,3),(2,4),(5,6),(5,7),(7,9),(6,8),(8,10),
               (5,11),(6,12),(11,12),(11,13),(13,15),(12,14),(14,16)]

def draw_text(img, text, x, y, color=(255,255,255), scale=0.5):
    font=cv2.FONT_HERSHEY_SIMPLEX
    (tw,th),base=cv2.getTextSize(text,font,scale,1)
    x=max(0,min(int(x),img.shape[1]-tw-6)); y=max(th+6,min(int(y),img.shape[0]-base-2))
    cv2.rectangle(img,(x,y-th-5),(x+tw+5,y+base+2),(20,20,20),-1)
    cv2.putText(img,text,(x+2,y),font,scale,color,1,cv2.LINE_AA)

def apply_segmentation(img, result, custom_boxes):
    if result.masks is None or result.boxes is None: return 0
    height,width=img.shape[:2]; overlay=img.copy(); applied=0
    masks=result.masks.data.cpu().numpy()
    classes=result.boxes.cls.int().cpu().tolist()
    scores=result.boxes.conf.cpu().numpy()
    for idx,(raw_mask, clsid, score) in enumerate(zip(masks,classes,scores)):
        name=str(result.names[clsid]).lower()
        if name=='person': continue
        mask=(raw_mask>0.5).astype(np.uint8)
        if mask.shape!=(height,width):
            mask=cv2.resize(mask,(width,height),interpolation=cv2.INTER_NEAREST)
        area=int(mask.sum())
        if not area: continue
        overlap=np.zeros((height,width),dtype=np.uint8)
        for x1,y1,x2,y2 in custom_boxes:
            x1=max(0,min(width,int(x1))); x2=max(0,min(width,int(x2)))
            y1=max(0,min(height,int(y1))); y2=max(0,min(height,int(y2)))
            overlap[y1:y2,x1:x2]=1
        if (mask & overlap).sum()/area>=MASK_OVERLAP_SUPPRESS: continue
        color=COLORES_SEG[clsid % len(COLORES_SEG)]
        overlay[mask.astype(bool)] = color
        contours,_=cv2.findContours(mask,cv2.RETR_EXTERNAL,cv2.CHAIN_APPROX_SIMPLE)
        cv2.drawContours(img,contours,-1,color,1)
        # Mostrar etiqueta cerca de la máscara, si es posible.
        if contours:
            contour=max(contours,key=cv2.contourArea)
            x,y,w,h=cv2.boundingRect(contour)
            draw_text(img,f'COCO {name} {score:.2f}',x,y-4,color)
        applied+=1
    blended=cv2.addWeighted(overlay,MASK_ALPHA,img,1-MASK_ALPHA,0)
    # Mantener contornos/etiquetas dibujados previamente en img.
    img[:]=blended
    return applied

def draw_pose(img, result):
    if result.keypoints is None: return 0
    points=result.keypoints.xy.cpu().numpy()
    confidence=(result.keypoints.conf.cpu().numpy() if result.keypoints.conf is not None
                else np.ones(points.shape[:2],dtype=float))
    track_ids=None
    if result.boxes is not None and result.boxes.id is not None:
        track_ids=result.boxes.id.int().cpu().numpy()
    for pi,(joints,probs) in enumerate(zip(points,confidence)):
        for a,b in COCO_SKELETON:
            if a>=len(joints) or b>=len(joints): continue
            if probs[a]<CONF_KEYPOINT or probs[b]<CONF_KEYPOINT: continue
            pa=tuple(map(int,joints[a])); pb=tuple(map(int,joints[b]))
            if min(*pa,*pb)<0: continue
            cv2.line(img,pa,pb,(230,65,240),2,cv2.LINE_AA)
        for (x,y),prob in zip(joints,probs):
            if prob>=CONF_KEYPOINT and x>0 and y>0:
                cv2.circle(img,(int(x),int(y)),3,(40,255,255),-1,cv2.LINE_AA)
        valid=np.where(probs>=CONF_KEYPOINT)[0]
        if len(valid):
            label='Persona' + (f' ID {track_ids[pi]}' if track_ids is not None else '')
            draw_text(img,label,int(joints[valid[0],0]),int(joints[valid[0],1])-10,(230,65,240))
    return len(points)

def draw_custom(img, result):
    count=Counter(); custom_boxes=[]
    if result.boxes is None: return count,custom_boxes
    for box,clsid,score in zip(result.boxes.xyxy.cpu().numpy(),
                               result.boxes.cls.int().cpu().tolist(),
                               result.boxes.conf.cpu().numpy()):
        name=str(result.names[clsid]).lower(); x1,y1,x2,y2=map(int,box)
        custom_boxes.append((x1,y1,x2,y2)); count[name]+=1
        color=COLORES_CUSTOM.get(name,(0,255,0))
        cv2.rectangle(img,(x1,y1),(x2,y2),color,2)
        draw_text(img,f'{name} {score:.2f}',x1,y1-6,color)
    return count,custom_boxes

## 5. Procesamiento: integrar las tres tareas sobre los mismos fotogramas
Se calcula inferencia secuencial sobre cada fotograma y se compone el resultado antes de guardarlo. El video final conserva dimensiones y FPS originales. La conversión a H.264/`yuv420p` permite reproducirlo de forma más consistente en Colab.

Nota técnica: los IDs del tracker indican asociación temporal, no garantizan la identidad perfecta bajo oclusiones complejas.

In [ ]:
def process_video(source:Path, destination:Path):
    cap=cv2.VideoCapture(str(source))
    fps=float(cap.get(cv2.CAP_PROP_FPS)); count=int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    width=int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)); height=int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    if fps<=0 or count<=0 or width<=0 or height<=0:
        cap.release(); raise RuntimeError(f'Video inválido: {source}')
    if count/fps>MAX_SECONDS:
        cap.release(); raise RuntimeError(f'Supera {MAX_SECONDS} segundos: {source}')
    raw=destination.with_name(destination.stem+'_temporal.mp4')
    writer=cv2.VideoWriter(str(raw),cv2.VideoWriter_fourcc(*'mp4v'),fps,(width,height))
    if not writer.isOpened(): cap.release(); raise RuntimeError('No se pudo crear video temporal.')
    detections=Counter(); seg_count=0; pose_count=0; processed=0
    # Nueva instancia de modelo pose por video: evita reutilizar estado del tracker entre videos.
    pose_model=YOLO(POSE_MODEL_NAME)
    start=time.time()
    try:
        while True:
            ret,frame=cap.read()
            if not ret: break
            custom=DETECTOR.predict(frame,conf=CONF_CUSTOM,imgsz=IMGSZ,device=DEVICE,verbose=False)[0]
            coco=SEGMENTADOR.predict(frame,conf=CONF_SEG,imgsz=IMGSZ,device=DEVICE,
                                     retina_masks=True,verbose=False)[0]
            human=pose_model.track(frame,conf=CONF_POSE,imgsz=IMGSZ,device=DEVICE,
                                   persist=True,tracker='bytetrack.yaml',verbose=False)[0]
            painted=frame.copy()
            # Boxes temporales para excluir máscaras superpuestas a las clases nuevas.
            custom_coords = (custom.boxes.xyxy.cpu().numpy().tolist()
                             if custom.boxes is not None else [])
            seg_count += apply_segmentation(painted,coco,custom_coords)
            pose_count += draw_pose(painted,human)
            class_count,_=draw_custom(painted,custom)
            detections.update(class_count)
            draw_text(painted,'YOLO26 | deteccion + segmentacion COCO + pose',10,24)
            writer.write(painted); processed+=1
            if processed%max(1,int(fps*5))==0:
                print(f'{source.name}: {processed}/{count} cuadros')
    finally:
        writer.release(); cap.release()
    if processed==0: raise RuntimeError('No se decodificaron fotogramas.')
    cmd=['ffmpeg','-y','-loglevel','error','-i',str(raw),'-c:v','libx264','-preset','veryfast',
         '-crf','23','-pix_fmt','yuv420p','-movflags','+faststart',str(destination)]
    try:
        subprocess.run(cmd,check=True,capture_output=True,text=True)
    except Exception as exc:
        shutil.move(raw,destination)
        print('Advertencia: conversión H264 no disponible, se conservó MP4 temporal:',exc)
    else:
        raw.unlink(missing_ok=True)
    return {'video':source.name,'salida':destination.name,'fotogramas':processed,
            'detecciones_mate':detections['mate'],'detecciones_termo':detections['termo'],
            'mascaras_COCO':seg_count,'poses_personas':pose_count,
            'segundos_ejecucion':round(time.time()-start,1)}

resultados=[]
for video in videos:
    output=VIDEOS_OUT/(video.stem+'_yolo26.mp4')
    resultados.append(process_video(video,output))
resumen=pd.DataFrame(resultados)
display(resumen)
resumen.to_csv(VIDEOS_OUT/'resumen_procesamiento.csv',index=False)


## 6. Inspección visual de resultados
Revisar manualmente los videos completos, no solamente un frame. A continuación se muestran el primer tercio de cada original y su resultado procesado para inspección rápida.

In [ ]:
def frame_at(path, fraction=0.33):
    cap=cv2.VideoCapture(str(path)); count=int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.set(cv2.CAP_PROP_POS_FRAMES,max(0,int(count*fraction)))
    ok,frame=cap.read(); cap.release()
    return cv2.cvtColor(frame,cv2.COLOR_BGR2RGB) if ok else None
for row in resultados:
    original=VIDEOS_IN/row['video']; processed=VIDEOS_OUT/row['salida']
    a,b=frame_at(original),frame_at(processed)
    if a is None or b is None: continue
    fig,ax=plt.subplots(1,2,figsize=(16,6))
    ax[0].imshow(a);ax[0].set_title('Original — '+original.name)
    ax[1].imshow(b);ax[1].set_title('Procesado — '+processed.name)
    for v in ax:v.axis('off')
    plt.tight_layout();plt.show()
    print('Video resultante:',processed)
    # Incrustar en Colab sólo si no supera 20 MB
    if processed.stat().st_size < 20_000_000:
        display(Video(str(processed),embed=True))

## 7. Control de cobertura y conclusiones
El total de detecciones representa **apariciones por fotograma**, no objetos únicos. Para demostrar la integración es deseable que el conjunto de videos produzca detecciones de ambas clases, máscaras COCO y poses de personas.

Si faltan detecciones, no asumir que los modelos funcionan: verificar primero la presencia real y visibilidad de los objetos en los videos. Puede necesitarse otro video o revisar el detector personalizado.

In [ ]:
if not resultados: raise RuntimeError('No hay videos procesados.')
totals=resumen[['detecciones_mate','detecciones_termo','mascaras_COCO','poses_personas']].sum()
print('Eventos observados por fotograma:')
print(totals.to_string())
missing=[k for k,v in totals.items() if v==0]
if missing:
    print('ATENCIÓN: no se observó evidencia visual de:',', '.join(missing))
    print('Elegir otros videos o revisar parámetros, etiquetas y modelo.')
else:
    print('Hay evidencia de las cuatro salidas esperadas; revisar la calidad de las anotaciones.')
print('Archivos finales en:',VIDEOS_OUT)

## 8. Entrega y cierre
Entregar **ambos notebooks**, el dataset entrenado con `data.yaml`, `best.pt`, documentación de orígenes y licencias, videos originales y procesados, y resultados/interpretación. En las conclusiones explicar aciertos, falsos positivos, limitaciones, confianza utilizada, tasa de cuadros procesados y posibles mejoras.

**No declarar validado el resultado solamente porque existe un MP4**: comprobar máscaras, esqueleto, nombre de clases y confianza en ejemplos reales.